# CSCN8020 Assignment 3: DQN on PongDeterministic-v4

| Item | Value |
|---|---|
| Student Name | Jatinder Pal Singh |
| Student ID | 9083762 |
| Course | CSCN8020 Reinforcement Learning |
| Date | March 28, 2026 |
| Repository | https://github.com/jatinder3762/DQN_Assignment3.git |

This notebook implements a Deep Q-Network (DQN) agent for PongDeterministic-v4 using OOP design and the required assignment utility file.

## Assignment Checklist

- Environment: PongDeterministic-v4
- Observation preprocessing from assignment3_utils.py
- 4 stacked grayscale frames as CNN input
- Hyperparameters: batch=8, target update every 10 episodes, gamma=0.95, epsilon init=1.0, decay=0.995, min=0.05
- Metrics vs steps: score per episode, average cumulative reward of last 5 episodes
- Parameter studies:
  - Batch size: 8 (default) vs 16
  - Target update rate: every 3 vs 10 episodes

Note: Full training in Pong is computationally heavy. This notebook includes a small default run for reproducibility and can be scaled up by increasing episodes.

In [ ]:
import os
import random
from collections import deque, namedtuple
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

try:
    import gymnasium as gym
except ImportError:
    import gym

from assignment3_utils import process_frame

print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())

In [ ]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
ENV_NAME = 'PongDeterministic-v4'

FRAME_H = 84
FRAME_W = 80
STACK_SIZE = 4
STATE_SHAPE_SINGLE = (FRAME_H, FRAME_W)

@dataclass
class Config:
    gamma: float = 0.95
    lr: float = 1e-4
    batch_size: int = 8
    target_update_episodes: int = 10
    eps_init: float = 1.0
    eps_decay: float = 0.995
    eps_min: float = 0.05
    replay_capacity: int = 50_000
    learn_start: int = 1000
    max_steps_per_episode: int = 5000
    episodes: int = 30

cfg_default = Config()
cfg_default

## OOP Components

In [ ]:
Transition = namedtuple('Transition', ['state', 'action', 'reward', 'next_state', 'done'])

class ReplayBuffer:
    def __init__(self, capacity: int):
        self.buffer = deque(maxlen=capacity)

    def push(self, state, action, reward, next_state, done):
        self.buffer.append(Transition(state, action, reward, next_state, done))

    def sample(self, batch_size: int):
        transitions = random.sample(self.buffer, batch_size)
        batch = Transition(*zip(*transitions))
        return batch

    def __len__(self):
        return len(self.buffer)

In [ ]:
class DQNCNN(nn.Module):
    def __init__(self, n_actions: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(STACK_SIZE, 32, kernel_size=8, stride=4),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1),
            nn.ReLU(),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, STACK_SIZE, FRAME_H, FRAME_W)
            n_flat = self.features(dummy).view(1, -1).shape[1]

        self.head = nn.Sequential(
            nn.Linear(n_flat, 512),
            nn.ReLU(),
            nn.Linear(512, n_actions),
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.head(x)

In [ ]:
class FrameStackProcessor:
    def __init__(self, stack_size: int = 4):
        self.stack_size = stack_size
        self.frames = deque(maxlen=stack_size)

    def reset(self, first_obs):
        frame = process_frame(first_obs, STATE_SHAPE_SINGLE).squeeze(0).squeeze(-1)
        self.frames.clear()
        for _ in range(self.stack_size):
            self.frames.append(frame)
        return self.get_state()

    def step(self, obs):
        frame = process_frame(obs, STATE_SHAPE_SINGLE).squeeze(0).squeeze(-1)
        self.frames.append(frame)
        return self.get_state()

    def get_state(self):
        stacked = np.stack(self.frames, axis=0).astype(np.float32)
        return stacked

In [ ]:
class DQNAgent:
    def __init__(self, n_actions: int, config: Config):
        self.n_actions = n_actions
        self.cfg = config

        self.policy_net = DQNCNN(n_actions).to(DEVICE)
        self.target_net = DQNCNN(n_actions).to(DEVICE)
        self.target_net.load_state_dict(self.policy_net.state_dict())
        self.target_net.eval()

        self.optimizer = optim.Adam(self.policy_net.parameters(), lr=self.cfg.lr)
        self.loss_fn = nn.MSELoss()

        self.replay = ReplayBuffer(self.cfg.replay_capacity)
        self.epsilon = self.cfg.eps_init

    def act(self, state: np.ndarray) -> int:
        if random.random() < self.epsilon:
            return random.randrange(self.n_actions)

        state_t = torch.from_numpy(state).unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            qvals = self.policy_net(state_t)
            return int(torch.argmax(qvals, dim=1).item())

    def remember(self, state, action, reward, next_state, done):
        self.replay.push(state, action, reward, next_state, done)

    def learn(self):
        if len(self.replay) < max(self.cfg.batch_size, self.cfg.learn_start):
            return None

        batch = self.replay.sample(self.cfg.batch_size)

        states = torch.from_numpy(np.array(batch.state)).float().to(DEVICE)
        actions = torch.tensor(batch.action, dtype=torch.long, device=DEVICE).unsqueeze(1)
        rewards = torch.tensor(batch.reward, dtype=torch.float32, device=DEVICE).unsqueeze(1)
        next_states = torch.from_numpy(np.array(batch.next_state)).float().to(DEVICE)
        dones = torch.tensor(batch.done, dtype=torch.float32, device=DEVICE).unsqueeze(1)

        q_values = self.policy_net(states).gather(1, actions)
        with torch.no_grad():
            max_next_q = self.target_net(next_states).max(1, keepdim=True)[0]
            target_q = rewards + self.cfg.gamma * max_next_q * (1.0 - dones)

        loss = self.loss_fn(q_values, target_q)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy_net.parameters(), 5.0)
        self.optimizer.step()
        return float(loss.item())

    def update_target(self):
        self.target_net.load_state_dict(self.policy_net.state_dict())

    def decay_epsilon(self):
        if self.epsilon >= self.cfg.eps_min:
            self.epsilon = self.epsilon * self.cfg.eps_decay
        else:
            self.epsilon = self.cfg.eps_min

In [ ]:
class DQNTrainer:
    def __init__(self, env_name: str, config: Config, seed: int = 42):
        self.env_name = env_name
        self.cfg = config
        self.seed = seed

        self.env = gym.make(env_name)
        self.n_actions = self.env.action_space.n
        self.agent = DQNAgent(self.n_actions, config)
        self.processor = FrameStackProcessor(STACK_SIZE)

        self.episode_scores = []
        self.episode_steps = []
        self.total_steps = 0

    def _reset_env(self):
        out = self.env.reset(seed=self.seed)
        if isinstance(out, tuple):
            obs, _ = out
        else:
            obs = out
        state = self.processor.reset(obs)
        return state

    def _step_env(self, action):
        out = self.env.step(action)
        if len(out) == 5:
            obs, reward, terminated, truncated, info = out
            done = terminated or truncated
        else:
            obs, reward, done, info = out
        return obs, float(reward), done, info

    def train(self, episodes: int = None, verbose_every: int = 1):
        if episodes is None:
            episodes = self.cfg.episodes

        scores = []
        avg5 = []
        step_axis = []

        for ep in range(1, episodes + 1):
            state = self._reset_env()
            done = False
            ep_score = 0.0
            ep_steps = 0

            while not done and ep_steps < self.cfg.max_steps_per_episode:
                action = self.agent.act(state)
                obs_next, reward, done, _ = self._step_env(action)
                next_state = self.processor.step(obs_next)

                self.agent.remember(state, action, reward, next_state, done)
                self.agent.learn()

                state = next_state
                ep_score += reward
                ep_steps += 1
                self.total_steps += 1

            self.agent.decay_epsilon()

            if ep % self.cfg.target_update_episodes == 0:
                self.agent.update_target()

            scores.append(ep_score)
            self.episode_scores.append(ep_score)
            self.episode_steps.append(ep_steps)

            avg_last5 = float(np.mean(scores[-5:]))
            avg5.append(avg_last5)
            step_axis.append(self.total_steps)

            if ep % verbose_every == 0:
                print(f'Episode {ep:03d} | steps={ep_steps:4d} | score={ep_score:6.1f} | avg5={avg_last5:6.2f} | eps={self.agent.epsilon:0.4f}')

        metrics = {
            'episode': np.arange(1, episodes + 1),
            'steps': np.array(step_axis),
            'score': np.array(scores, dtype=float),
            'avg_last_5': np.array(avg5, dtype=float),
        }
        return metrics

    def close(self):
        self.env.close()

## Training Helpers

In [ ]:
def plot_metrics(metrics, title_prefix='Run'):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    axes[0].plot(metrics['steps'], metrics['score'], marker='o', ms=3)
    axes[0].set_title(f'{title_prefix} - Score per Episode')
    axes[0].set_xlabel('Training Steps')
    axes[0].set_ylabel('Score')

    axes[1].plot(metrics['steps'], metrics['avg_last_5'], marker='o', ms=3, color='darkorange')
    axes[1].set_title(f'{title_prefix} - Average Cumulative Reward (Last 5 Episodes)')
    axes[1].set_xlabel('Training Steps')
    axes[1].set_ylabel('Average Reward')

    plt.tight_layout()
    plt.show()


def run_experiment(config: Config, episodes: int, label: str):
    trainer = DQNTrainer(ENV_NAME, config=config, seed=SEED)
    metrics = trainer.train(episodes=episodes, verbose_every=max(1, episodes // 5))
    trainer.close()
    plot_metrics(metrics, title_prefix=label)
    return metrics

## Default Run (Required Hyperparameters)

The next cell runs with the assignment defaults:
- batch size = 8
- target update every 10 episodes
- gamma = 0.95
- epsilon: 1.0, decay 0.995, min 0.05

In [ ]:
default_episodes = 20  # increase to 300+ for stronger Pong performance
metrics_default = run_experiment(cfg_default, episodes=default_episodes, label='Default (batch=8, target=10)')

## Parameter Study 1: Mini-batch Size (8 vs 16)

In [ ]:
episodes_study = 20

cfg_batch8 = Config(batch_size=8, target_update_episodes=10)
cfg_batch16 = Config(batch_size=16, target_update_episodes=10)

metrics_b8 = run_experiment(cfg_batch8, episodes=episodes_study, label='Batch 8')
metrics_b16 = run_experiment(cfg_batch16, episodes=episodes_study, label='Batch 16')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(metrics_b8['steps'], metrics_b8['score'], label='batch=8')
axes[0].plot(metrics_b16['steps'], metrics_b16['score'], label='batch=16')
axes[0].set_title('Score per Episode vs Steps')
axes[0].set_xlabel('Training Steps')
axes[0].set_ylabel('Score')
axes[0].legend()

axes[1].plot(metrics_b8['steps'], metrics_b8['avg_last_5'], label='batch=8')
axes[1].plot(metrics_b16['steps'], metrics_b16['avg_last_5'], label='batch=16')
axes[1].set_title('Avg Cumulative Reward (Last 5 Episodes) vs Steps')
axes[1].set_xlabel('Training Steps')
axes[1].set_ylabel('Average Reward')
axes[1].legend()

plt.tight_layout()
plt.show()

## Parameter Study 2: Target Update Rate (3 vs 10 episodes)

In [ ]:
cfg_t3 = Config(batch_size=8, target_update_episodes=3)
cfg_t10 = Config(batch_size=8, target_update_episodes=10)

metrics_t3 = run_experiment(cfg_t3, episodes=episodes_study, label='Target update=3 episodes')
metrics_t10 = run_experiment(cfg_t10, episodes=episodes_study, label='Target update=10 episodes')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(metrics_t3['steps'], metrics_t3['score'], label='update=3')
axes[0].plot(metrics_t10['steps'], metrics_t10['score'], label='update=10')
axes[0].set_title('Score per Episode vs Steps')
axes[0].set_xlabel('Training Steps')
axes[0].set_ylabel('Score')
axes[0].legend()

axes[1].plot(metrics_t3['steps'], metrics_t3['avg_last_5'], label='update=3')
axes[1].plot(metrics_t10['steps'], metrics_t10['avg_last_5'], label='update=10')
axes[1].set_title('Avg Cumulative Reward (Last 5 Episodes) vs Steps')
axes[1].set_xlabel('Training Steps')
axes[1].set_ylabel('Average Reward')
axes[1].legend()

plt.tight_layout()
plt.show()

## Network Architecture Used

Input: 4 x 84 x 80 stacked grayscale frames

CNN:
1. Conv2d(4, 32, kernel=8, stride=4) + ReLU
2. Conv2d(32, 64, kernel=4, stride=2) + ReLU
3. Conv2d(64, 64, kernel=3, stride=1) + ReLU
4. Flatten
5. Linear(flat, 512) + ReLU
6. Linear(512, n_actions)

Loss: Mean Squared Error between current Q and target Q
Optimizer: Adam (lr=1e-4)
Discount factor: gamma=0.95

## Observations and Best Hyperparameter Choice

After running the parameter sweeps:
- Batch size 8 usually learns faster early than 16 in low-data settings because updates happen with less delay and more gradient noise, which can help exploration.
- Target update every 10 episodes is often more stable than every 3 episodes because targets change less frequently.

Selected best combination for this assignment:
- mini-batch size = 8
- target update every 10 episodes

This choice follows the default assignment hyperparameters and generally gives good stability for Pong DQN.

Note: Final best values can vary by random seed and training budget; for final report quality, run 3 seeds and average the curves.